# 🧪 W1-D1 概念实验：自注意力机制详解

> 配套阅读：`第1周-Day1-自注意力机制详解.md`（公式推导与文字讲解在那边）
> 这个 notebook 只做一件事：**用可执行的小实验把 Q/K/V、√d_k 缩放、动态加权这几个概念"跑"出来**
>
> 实验环境：纯 numpy + matplotlib，无需 GPU、无需网络。

## 实验 1：手算一遍 Q/K/V —— 注意力权重矩阵长什么样？

句子 "我 爱 AI"，3 个词、每个词 4 维向量。跑完整流程：
投影 → 打分 → 缩放 → softmax → 加权求和。重点看那个 3×3 权重矩阵：**每一行都是一个概率分布（和为 1），第 i 行 = 第 i 个词把注意力分给了谁**。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# 模拟句子 "我 爱 AI"：3 个词，每个 4 维词向量（真实模型中来自 Embedding 层）
X = rng.normal(size=(3, 4))

# 三个投影矩阵（真实模型中是训练出来的，这里随机初始化，先看机制）
W_q = rng.normal(scale=0.5, size=(4, 4))
W_k = rng.normal(scale=0.5, size=(4, 4))
W_v = rng.normal(scale=0.5, size=(4, 4))

Q, K, V = X @ W_q, X @ W_k, X @ W_v

d_k = Q.shape[1]
scores = Q @ K.T / np.sqrt(d_k)                     # ① 缩放点积打分

e = np.exp(scores - scores.max(axis=1, keepdims=True))  # ② softmax（减最大值防溢出）
weights = e / e.sum(axis=1, keepdims=True)
out = weights @ V                                   # ③ 加权求和

tokens = ["我", "爱", "AI"]
print("注意力权重矩阵（行 = 查询词，列 = 被看的词）：")
print("      " + "  ".join(f"{t:>5}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>4}  " + "  ".join(f"{v:5.3f}" for v in row))
print()
print("每行和 =", weights.sum(axis=1).round(6), " ← softmax 保证每行是概率分布")
print("输出 shape =", out.shape, "← 与输入相同，所以能一层层堆叠 96 层")

## 实验 2：为什么必须除以 √d_k？

如果 q、k 的每个分量都是标准正态，点积 q·k 的方差是 d_k。
d_k=512 时分数的标准差 ≈ 22.6 —— softmax 遇到这么大的分差会**饱和成 one-hot**：赢者通吃、梯度消失。
除以 √512 后分数回到 N(0,1)，分布温和。用熵来量化这件事。

In [ ]:
d_k = 512

# 采样 2000 个 q·k 点积，看未缩放时的分布
q = rng.normal(size=d_k)
dots = np.array([q @ rng.normal(size=d_k) for _ in range(2000)])
print(f"未缩放分数：均值 {dots.mean():+.1f}，标准差 {dots.std():.1f}"
      f"（理论值 √d_k = {np.sqrt(d_k):.1f}）")

def softmax(s):
    e = np.exp(s - s.max())
    return e / e.sum()

def entropy(p):   # 熵越大 = 注意力越"分散"；熵 = 0 = 只看一个词
    return -(p * np.log(p + 1e-12)).sum()

row = rng.normal(size=d_k) * dots.std()     # 一行典型分数（未缩放）
row_scaled = row / np.sqrt(d_k)

p_raw, p_scaled = softmax(row), softmax(row_scaled)
print(f"\n未缩放：最大权重 {p_raw.max():.4f}，熵 {entropy(p_raw):.3f}（≈one-hot，其余词的信息进不来）")
print(f"缩放后：最大权重 {p_scaled.max():.4f}，熵 {entropy(p_scaled):.3f}（分布温和，多词信息都能保留）")
print(f"\n最大熵上限 ln({d_k}) = {np.log(d_k):.2f}")

## 实验 3：Q·K 决定"看哪里" —— 手工操控注意力

不训练，直接手工构造 Q 和 K：让代词"它"的 query 与"猫"的 key 同向，
其余随机。看第 4 行（"它"）的注意力是不是自动集中到"猫"上——
这证明了 **注意力权重完全由 Q·K 的几何对齐程度决定**，训练做的事就是调 W_q/W_k 让对齐变得"有意义"。

In [ ]:
tokens3 = ["猫", "追", "球", "，", "它", "累"]
n, d = len(tokens3), 8

Q3 = rng.normal(size=(n, d))
K3 = rng.normal(size=(n, d))
V3 = rng.normal(size=(n, d))

# 手工设定：让 "它"(i=4) 的 query 指向 "猫"(j=0) 的 key
Q3[4] = 2.0 * K3[0]

s = Q3 @ K3.T / np.sqrt(d)
w3 = np.apply_along_axis(softmax, 1, s)

print("      " + "  ".join(f"{t:>4}" for t in tokens3))
for t, row in zip(tokens3, w3):
    bar = "█" * int(row.max() * 40)
    print(f"{t:>4}  " + "  ".join(f"{v:4.2f}" for v in row) + f"   最关注: {tokens3[int(row.argmax())]} {bar}")
print("\n→ 「它」 的 query 与 「猫」 的 key 同向，点积最大，注意力自动聚焦（指代消歧就是这样发生的）")

## 实验 4：缩放前 vs 缩放后的注意力热力图

8 个词、d_k=64。同一组 Q/K，左边不缩放、右边除以 √64=8。
未缩放的热力图每一行几乎只有一个亮点（饱和）；缩放后行内出现有层次的分布。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

n2, d2 = 8, 64
X2 = rng.normal(size=(n2, d2))
Wq2 = rng.normal(scale=0.6, size=(d2, d2))
Wk2 = rng.normal(scale=0.6, size=(d2, d2))
Q2, K2 = X2 @ Wq2, X2 @ Wk2

raw = Q2 @ K2.T
scaled = raw / np.sqrt(d2)

def row_softmax(m):
    e = np.exp(m - m.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

labels = [f"词{i}" for i in range(n2)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, mat, title in [
    (axes[0], row_softmax(raw),      f"未缩放（每行最大权重平均 {row_softmax(raw).max(axis=1).mean():.3f}）"),
    (axes[1], row_softmax(scaled),   f"缩放 /√{d2}（每行最大权重平均 {row_softmax(scaled).max(axis=1).mean():.3f}）"),
]:
    im = ax.imshow(mat, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(n2), labels, rotation=45)
    ax.set_yticks(range(n2), labels)
    ax.set_xlabel("被看的词 (K)")
    ax.set_ylabel("查询的词 (Q)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("√d_k 缩放前后：注意力分布从饱和 one-hot 变为有层次")
plt.tight_layout()
plt.show()

## 实验 5：全局感受野 —— 换一个词，全句的输出都变

自注意力 vs CNN/RNN 最大的结构差异：**任何两个词之间都是一步直达**。
把"爱"换成一个完全不同的词，重新算权重矩阵——不只"爱"那一行，**每一行**的注意力都会改变。

In [ ]:
X_b = X.copy()
X_b[1] = rng.normal(size=4) * 5.0      # 把第 2 个词换成完全不同的向量

Qb, Kb, Vb = X_b @ W_q, X_b @ W_k, X_b @ W_v
sb = Qb @ Kb.T / np.sqrt(d_k)
eb = np.exp(sb - sb.max(axis=1, keepdims=True))
w_b = eb / eb.sum(axis=1, keepdims=True)

print("换词前权重：\n", np.round(weights, 3))
print("换词后权重：\n", np.round(w_b, 3))
print("\n每一行的平均变化量：", np.abs(w_b - weights).mean(axis=1).round(4))
print("→ 只改了 1 个词，3 行权重全部变化；输出 out 的每一行都受影响（全局感受野）")

## 结论

| 概念 | 实验验证 |
|---|---|
| softmax 权重 | 每行和恒为 1，是"注意力预算"的分配（实验 1） |
| √d_k 缩放 | 未缩放熵≈0（one-hot 饱和），缩放后保留多词信息（实验 2、4） |
| Q·K 对齐 | 手工对齐 query/key 即可操控注意力落点（实验 3） |
| 全局感受野 | 改 1 个词 → 全部输出变化；一步直达，无距离衰减（实验 5） |

→ 深入阅读：同目录 `.md` 版本第二节（完整公式推导 + 2.6 完整公式拆解）